# Revision Walkthrough: Corrected Constructions and Explicit Costs

This notebook demonstrates the work completed so far for the revision of
"From Graph $k$-Coloring to Grover's Circuits via Logic Synthesis". It follows
one small instance, the triangle $K_3$ with three colors and no precolored
vertex, from its coloring predicate to explicitly decomposed circuits and their
costs, and it draws the intermediate representations along the way.

The sections cover:

1. the two coloring predicates and their Xor-And-Inverter Graphs (XAGs);
2. the synthesized oracle circuit and its verification;
3. one Grover iteration whose diffusion operator reflects about the actual
   preparation, compared with the historical Hadamard reflection;
4. the vertex-labelling issue of the Saha et al. baseline;
5. explicit decompositions of multi-controlled X gates and the exact
   Clifford+T gadgets used for counting;
6. relative-phase Toffoli gates in mirrored compute blocks;
7. measurement-based uncomputation of temporary logical-AND gates;
8. the fault-tolerant cost of one complete Grover iteration;
9. routed metrics on the `FakeTorino` device model.

Block-level drawings, in which a relative-phase Toffoli or a doubly controlled
$R_x(\pm\pi)$ appears as one box, are followed by expanded drawings of the
exact Clifford+T gates that the cost model counts. These expansions are the
gadgets of Section 8; Qiskit's own definitions of the same blocks differ (its
relative-phase Toffoli, for example, has T-depth 4 rather than 2).

All numbers are computed in this run with an unpatched Qiskit. The historical
replay of the archived paper row is kept separately in
`illustrative_example_v2.ipynb`, which needs the replay environment.

## 1. Set up the run

The notebook runner supplies `LOGIQUE_OUTPUT`; an interactive run creates a new
folder under `dump/`. Figures are saved as PNG and SVG files in that folder.

In [ ]:
from pathlib import Path
import json
import math
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import SVG, Image, Markdown, display
from qiskit import QuantumCircuit

from logique.benchmarks import costs
from logique.paths import native_executable, new_dump_directory

ROOT = Path(os.environ.get("LOGIQUE_WORKSPACE", Path.cwd())).resolve()
if "LOGIQUE_OUTPUT" in os.environ:
    OUT = Path(os.environ["LOGIQUE_OUTPUT"]).resolve()
else:
    OUT = new_dump_directory("revision-walkthrough", root=ROOT / "dump")
FIGURES = OUT / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)
EXECUTABLE = native_executable(
    os.environ.get("LOGIQUE_NATIVE_EXECUTABLE")
    or ROOT / "external/native/boolean_synthesis/build/boolean_synthesis"
)
plt.rcParams.update({"figure.dpi": 110, "font.size": 9})
pd.set_option("display.max_columns", 30)


def show_figure(figure, stem: str) -> None:
    """Save a Matplotlib figure as PNG and SVG, then display it."""
    figure.savefig(FIGURES / f"{stem}.svg", bbox_inches="tight")
    figure.savefig(FIGURES / f"{stem}.png", dpi=140, bbox_inches="tight")
    display(Image(filename=str(FIGURES / f"{stem}.png")))
    plt.close(figure)


def show_expanded(circuit, stem: str, fold: int = 40) -> None:
    """Draw the exact Clifford+T gates that the cost model counts."""
    lowered = costs.clifford_t(circuit)
    counts = costs.ft_metrics(lowered) if "if_else" not in lowered.count_ops() else None
    if counts is not None:
        display(
            Markdown(
                f"Expanded: {counts['t_count']} T gates, T-depth {counts['t_depth']:.0f}, "
                f"{counts['cx_count']} CNOTs, {counts['rotation_count']} rotations."
            )
        )
    show_figure(lowered.draw("mpl", fold=fold, idle_wires=False), stem + "-expanded")


BLOCK_LABELS = {"ccx": "Toffoli", "rccx": "RCCX", "rccx_dg": "RCCX†", "ccrx": "CC-Rx(±π)"}


def show_blockwise(circuit, stem: str, fold: int = 60) -> None:
    """Draw each block gate as its counted Clifford+T gadget between labelled barriers."""
    expanded = circuit.copy_empty_like()
    for index, inst in enumerate(circuit.data):
        label = BLOCK_LABELS.get(inst.operation.name)
        if label is None:
            expanded.append(inst.operation, inst.qubits, inst.clbits)
            continue
        block = QuantumCircuit(len(inst.qubits))
        block.append(inst.operation, range(len(inst.qubits)))
        expanded.barrier(label=label)
        expanded.compose(costs.clifford_t(block), inst.qubits, inplace=True)
        following = circuit.data[index + 1].operation.name if index + 1 < len(circuit.data) else None
        if following not in BLOCK_LABELS:
            expanded.barrier()
    counts = costs.ft_metrics(costs.clifford_t(circuit))
    display(
        Markdown(
            f"Expanded block by block: {counts['t_count']} T gates, T-depth "
            f"{counts['t_depth']:.0f}, {counts['cx_count']} CNOTs."
        )
    )
    show_figure(expanded.draw("mpl", fold=fold, idle_wires=False), stem + "-blocks")


def show_svg(svg: bytes, stem: str) -> None:
    """Save and display an SVG drawing produced by Graphviz."""
    (FIGURES / f"{stem}.svg").write_bytes(svg)
    display(SVG(data=svg))


print("Artifacts:", OUT)

## 2. The instance and its two predicates

Each vertex receives $c = \lceil \log_2 3 \rceil = 2$ qubits, least-significant
bit first, so the code $11$ (color 3) is unused. Two predicates are compared.

* The **complete predicate** requires adjacent vertices to differ and rejects the
  unused code at every vertex. It is correct for any preparation, including a
  Hadamard state over all codes.
* The **domain-free predicate** checks the edge inequalities only. It is the
  paper's edge-only predicate and is correct only when the preparation never
  produces the unused code, as the valid-color preparation guarantees. It is
  obtained here by encoding the same graph with four colors.

In [ ]:
from dataclasses import replace

import networkx as nx

from logique import encode_coloring, read_dimacs
from logique.visualization.graphs import draw_colored_graph

source = read_dimacs(ROOT / "datasets/graphs/benchmarks/K3.col")
problem = replace(source, precolored={})
print("Removed precolors:", dict(source.precolored))
graph = problem.to_networkx()
draw_colored_graph(
    graph,
    name=str(FIGURES / "k3-graph.png"),
    pos=nx.circular_layout(graph),
    figsize=(3.5, 3),
    node_size=1400,
)
plt.gcf().axes[0].set_title("K3 with three colors, no precolor")
show_figure(plt.gcf(), "k3-graph")

In [ ]:
complete_encoding = encode_coloring(problem)
domain_free_encoding = encode_coloring(replace(problem, colors=4))
labels = [None] * complete_encoding.input_count
for vertex, wires in complete_encoding.input_mapping.items():
    for bit, wire in enumerate(wires):
        labels[wire] = f"x_{vertex},{bit}"

assignments = [
    domain_free_encoding.decode(basis)
    for basis in range(1 << complete_encoding.input_count)
]
uses_unused_code = np.array([any(c >= 3 for c in a.values()) for a in assignments])
proper = np.array([problem.is_valid(a) for a in assignments])
edge_only = np.array([domain_free_encoding.problem.is_valid(a) for a in assignments])
display(
    pd.DataFrame(
        {
            "accepted assignments": [int(proper.sum()), int(edge_only.sum())],
            "of which use the unused code": [
                int((proper & uses_unused_code).sum()),
                int((edge_only & uses_unused_code).sum()),
            ],
        },
        index=["complete predicate", "domain-free predicate"],
    )
)

## 3. From predicate to Xor-And-Inverter Graph

Both predicates are written as structural Verilog and synthesized by the native
bridge into XAGs, which `xag_synth` (Meuli et al.) then maps to a quantum
circuit. The AND count of an XAG fixes the number of Toffoli-type gates. The
domain-free predicate needs $c - 1 = 1$ AND per edge and two more to combine the
three edges, five in total; the complete predicate adds the unused-code checks.

In [ ]:
from logique.synthesis import synthesize

domain_free = synthesize(domain_free_encoding, "xag", executable=EXECUTABLE)
complete = synthesize(complete_encoding, "xag", executable=EXECUTABLE)
summary = pd.DataFrame(
    {
        "domain-free": domain_free.summary(),
        "complete": complete.summary(),
    }
).loc[["qubits", "workspace", "AND", "XOR", "gates", "depth", "gate_types"]]
display(summary)

In [ ]:
from logique.visualization.boolean import graphviz_graph

for title, result, stem in (
    ("Domain-free predicate (edge inequalities only)", domain_free, "xag-domain-free"),
    ("Complete predicate (edges and unused codes)", complete, "xag-complete"),
):
    dot = graphviz_graph(result.metadata["network"], title, input_labels=labels)
    show_svg(dot.pipe(format="svg"), stem)

## 4. The synthesized oracle and its verification

The raw circuit holds the six data qubits, one output target and the workspace
that stores intermediate ANDs. Intermediate ANDs are computed with a doubly
controlled $R_x(\pi)$ and cleaned with $R_x(-\pi)$: their phases cancel in pairs.
The AND that drives the output is an exact Toffoli. For phase kick-back with the
output in $|-\rangle$, the raw circuit must act as $|x\rangle|y\rangle \mapsto
|x\rangle|y \oplus f(x)\rangle$ for both target values while restoring its
workspace; the check below establishes this for all 64 inputs.

In [ ]:
inputs = domain_free.metadata["input_qubits"]
outputs = domain_free.metadata["output_qubits"]
workspace = sorted(set(range(domain_free.circuit.num_qubits)) - set(inputs) - set(outputs))
display(
    pd.DataFrame(
        [
            {"role": "data", "qubits": inputs},
            {"role": "output target", "qubits": outputs},
            {"role": "workspace (clean, restored)", "qubits": workspace},
        ]
    )
)
show_figure(
    domain_free.circuit.draw("mpl", fold=30, idle_wires=False),
    "oracle-domain-free-raw",
)
show_expanded(domain_free.circuit, "oracle-domain-free-raw")

In [ ]:
from logique.verification import verify_kickback_oracle, verify_phase_oracle

checks = {
    "domain-free, kick-back": verify_kickback_oracle(
        domain_free, encoding=domain_free_encoding
    ),
    "complete, phase oracle": verify_phase_oracle(complete, encoding=complete_encoding),
}
display(pd.DataFrame(checks).T)
assert all(check["status"] == "exhaustive" for check in checks.values())

## 5. One Grover iteration that reflects about its own preparation

Each vertex is prepared in $|s_3\rangle = (|00\rangle + |01\rangle + |10\rangle)/\sqrt3$
by the Shukla–Vedula circuit (an $X$, one $R_y$ rotation and a controlled
Hadamard). With the domain-free oracle, the diffusion operator must reflect
about this preparation $A$, as $A(2|0\rangle\langle 0| - I)A^\dagger$. The archived
generator instead reflected about the Hadamard state, which leaks probability
into the unused code.

In [ ]:
from qiskit import QuantumCircuit
from qiskit.circuit.library.data_preparation.state_preparation import (
    UniformSuperpositionGate,
)

vertex_preparation = QuantumCircuit(2)
vertex_preparation.append(UniformSuperpositionGate(3), [0, 1])
vertex_preparation = vertex_preparation.decompose()
preparation = QuantumCircuit(len(inputs))
for start in range(0, len(inputs), 2):
    preparation.compose(vertex_preparation, [start, start + 1], inplace=True)
hadamard = QuantumCircuit(len(inputs))
hadamard.h(range(len(inputs)))
show_figure(vertex_preparation.draw("mpl"), "uqs-three-colors")
# The controlled Hadamard becomes two T gates; the Ry rotation stays a rotation.
show_expanded(vertex_preparation, "uqs-three-colors")

In [ ]:
from qiskit.quantum_info import Statevector

from logique.circuits.grover import kickback_grover_circuit

paper = kickback_grover_circuit(domain_free, preparation=preparation, iterations=1)
historical = kickback_grover_circuit(
    domain_free,
    preparation=preparation,
    reflection_preparation=hadamard,
    iterations=1,
)
show_figure(paper.draw("mpl", fold=40, idle_wires=False), "grover-iteration")

groups = {
    "proper coloring": proper,
    "valid codes, conflict": ~uses_unused_code & ~proper,
    "uses the unused code": uses_unused_code,
}
initial = Statevector(preparation).probabilities()
rows = []
for label, mask in groups.items():
    row = {"group": label, "initial": float(initial[mask].sum())}
    for name, circuit in (("paper reflection", paper), ("Hadamard reflection", historical)):
        row[name] = float(Statevector(circuit).probabilities(inputs)[mask].sum())
    rows.append(row)
probabilities = pd.DataFrame(rows).set_index("group")
display(probabilities.round(4))
figure, ax = plt.subplots(figsize=(7, 3.2))
probabilities.plot.bar(ax=ax, rot=0, color=["#86b6ef", "#2a78d6", "#eb6834"], width=0.75)
ax.set_ylabel("Probability after one iteration")
ax.set_xlabel("")
ax.grid(axis="y", color="#e4e3df")
ax.spines[["top", "right"]].set_visible(False)
show_figure(figure, "reflection-probabilities")

## 6. Baselines and vertex labels

The baselines come from the `saha_belletti` package. Its `original` variant
(Saha et al.) loops over vertex indices $0, \dots, n-1$ and uses each label as a
register index, so the one-based labels of the historical replay drop every
edge of the last vertex. The Belletti variants map labels to indices. The table
gives the ideal probability of the two proper 2-colorings of the three-vertex
path after one iteration, with the historical and the normalized labels; random
guessing gives 0.25.

In [ ]:
from logique.synthesis.saha_belletti import VARIANTS, comparison_circuit

path = read_dimacs(ROOT / "datasets/graphs/execution/linear3.col")
alternating = (0b010, 0b101)
rows = []
for variant in VARIANTS:
    row = {"variant": variant}
    for name, preserve in (("historical labels", True), ("normalized labels", False)):
        circuit = comparison_circuit(path, variant, iterations=1, preserve_labels=preserve)
        marginal = Statevector(circuit).probabilities([0, 1, 2])
        row[name] = float(sum(marginal[b] for b in alternating))
    rows.append(row)
labelling = pd.DataFrame(rows).set_index("variant")
display(labelling.round(4))
figure, ax = plt.subplots(figsize=(6.5, 3))
labelling.plot.bar(ax=ax, rot=0, color=["#eb6834", "#2a78d6"], width=0.7)
ax.axhline(0.25, color="#52514e", linewidth=1, label="random guess")
ax.set_ylabel("Ideal success, one iteration")
ax.set_xlabel("")
ax.grid(axis="y", color="#e4e3df")
ax.spines[["top", "right"]].set_visible(False)
ax.legend(frameon=False, ncol=3, loc="upper center", bbox_to_anchor=(0.5, -0.12))
show_figure(figure, "baseline-labels")
show_figure(
    comparison_circuit(path, "original", iterations=1).draw("mpl", fold=40),
    "saha-original-path3",
)

## 7. Explicit decompositions of multi-controlled X gates

Every gate with three or more controls is expanded by a named construction
before counting. Three constructions are drawn for four controls: the
throwaway tree of the manuscript's Figure 1, which leaves its two ancillas in a
garbage state; the restoring tree, which uncomputes its internal nodes with
relative-phase Toffoli gates; and the two-clean-ancilla construction of
Khattar and Gidney. For four controls, Qiskit's Khattar–Gidney circuit touches
only one of its two ancillas, and only that one is counted. The throwaway tree
is valid only inside a block that is later inverted, which the following check
demonstrates.

Each construction is drawn twice. The first drawing shows its blocks, and the
second replaces every block by the Clifford+T gadget that the cost model counts
(Section 8), between barriers labelled with the block's name: a Toffoli costs
seven T gates and a relative-phase Toffoli (RCCX) or its inverse (RCCX†) four.
Gates outside a labelled segment, such as the X gates of the Khattar–Gidney
construction, are drawn unchanged.

In [ ]:
gate = QuantumCircuit(5)
gate.mcx([0, 1, 2, 3], 4)
for method, stem in (
    ("throwaway_tree", "mcx-throwaway-tree"),
    ("clean_tree", "mcx-restoring-tree"),
    ("2_clean_kg24", "mcx-khattar-gidney"),
):
    expanded, info = costs.decompose_mcx(gate, method)
    display(Markdown(f"**{method}**: {info['mcx_reference']}"))
    show_figure(expanded.draw("mpl", fold=40), stem)
    show_blockwise(expanded, stem)

In [ ]:
from qiskit.quantum_info import Operator

block = QuantumCircuit(6)
block.mcx([0, 1, 2, 3], 4)  # compute an AND of four wires into wire 4
compute, info = costs.decompose_mcx(block, "throwaway_tree")
mirrored = compute.copy()
mirrored.z(4)
mirrored.compose(compute.inverse(), inplace=True)
reference = block.copy()
reference.z(4)
reference.compose(block.inverse(), inplace=True)
padded = QuantumCircuit(mirrored.num_qubits)
padded.compose(reference, qubits=range(6), inplace=True)
clean_columns = [i for i in range(2**mirrored.num_qubits) if i >> 6 == 0]
alone = Operator(compute).data
display(
    pd.DataFrame(
        {
            "garbage left by the compute block alone": [
                bool(np.abs(alone[2**6 :, clean_columns]).max() > 1e-9)
            ],
            "mirrored block equals the ideal oracle": [
                bool(
                    np.allclose(
                        Operator(mirrored).data[:, clean_columns],
                        Operator(padded).data[:, clean_columns],
                    )
                )
            ],
            "held ancillas": [info["garbage_ancillas"]],
        }
    )
)

In [ ]:
from logique.benchmarks import logical_and

rows = [
    costs.isolated_mcx_costs(method, n)
    for method in costs.MCX_DECOMPOSITIONS
    for n in (4, 8)
]
rows += [logical_and.isolated_mcx_assisted_costs("clean_tree", n) for n in (4, 8)]
isolated = pd.DataFrame(rows)
display(
    isolated.set_index(["controls", "method"])[
        ["workspace", "ancillas", "t_count", "t_depth", "rotation_count", "cx_count"]
    ].sort_index()
)
isolated.to_csv(OUT / "isolated_mcx_costs.csv", index=False)

## 8. Exact Clifford+T gadgets

Counting uses fixed gadgets whose T counts are exact. A Toffoli is
$H \cdot \mathrm{CCZ} \cdot H$, and CCZ is the phase polynomial
$\frac{\pi}{4}(a + b + c - a{\oplus}b - a{\oplus}c - b{\oplus}c + a{\oplus}b{\oplus}c)$:
seven T gates in three layers. A doubly controlled $R_z(\pi)$ needs only the four
terms $t,\ c_1{\oplus}t,\ c_2{\oplus}t,\ c_1{\oplus}c_2{\oplus}t$, which gives the
controlled $R_x(\pm\pi)$ of the XAG flow and the relative-phase Toffoli four T
gates in two layers. Each gadget below is checked against Qiskit's matrix.

In [ ]:
from qiskit.circuit.library import CCXGate, CHGate, RCCXGate, RXGate

rows = []
for name, gate in (
    ("Toffoli", CCXGate()),
    ("controlled Rx(pi)", RXGate(math.pi).control(2)),
    ("relative-phase Toffoli", RCCXGate()),
    ("controlled Hadamard", CHGate()),
):
    circuit = QuantumCircuit(gate.num_qubits)
    circuit.append(gate, range(gate.num_qubits))
    lowered = costs.clifford_t(circuit)
    metrics = costs.ft_metrics(lowered)
    rows.append(
        {
            "gate": name,
            "T-count": metrics["t_count"],
            "T-depth": metrics["t_depth"],
            "CNOT": metrics["cx_count"],
            "exact": bool(np.allclose(Operator(lowered).data, Operator(circuit).data)),
        }
    )
    show_figure(lowered.draw("mpl", fold=40), "gadget-" + name.replace(" ", "-").lower())
display(pd.DataFrame(rows).set_index("gate"))

## 9. Relative-phase Toffoli gates in mirrored blocks

When a compute block is later undone by its exact inverse, and only operations
that are diagonal in the computational basis act in between, the phases of
relative-phase Toffoli gates cancel. The XAG flow already exploits this, and
`relative_phase_toffolis()` gives the mirrored compute blocks of other oracles
the same four-T Toffoli gates. The small block below computes two ANDs and
combines them, as a comparator-based oracle does.

In [ ]:
block = QuantumCircuit(7)
block.ccx(0, 1, 4)
block.ccx(2, 3, 5)
block.ccx(4, 5, 6)


def mirrored(compute):
    oracle = compute.copy()
    oracle.z(6)
    oracle.compose(compute.inverse(), inplace=True)
    return oracle


relative = costs.relative_phase_toffolis(block)
display(
    pd.DataFrame(
        {
            "T-count of the mirrored oracle": [
                costs.ft_metrics(costs.clifford_t(mirrored(block)))["t_count"],
                costs.ft_metrics(costs.clifford_t(mirrored(relative)))["t_count"],
            ],
            "equal to the exact oracle": [
                True,
                bool(np.allclose(Operator(mirrored(relative)).data, Operator(mirrored(block)).data)),
            ],
        },
        index=["exact Toffoli gates", "relative-phase Toffoli gates"],
    )
)
show_figure(mirrored(relative).draw("mpl", fold=40), "mirrored-relative-phase")
show_expanded(mirrored(relative), "mirrored-relative-phase", fold=45)

## 10. Measurement-based uncomputation of logical-AND gates

Gidney's temporary logical-AND (arXiv:1709.06648) computes the AND of two qubits
into a clean qubit with four T gates. Uncomputation then needs no T gate:
measure the target in the X basis and, on outcome 1, apply a CZ to the controls
and reset the target. A pair qualifies when its target is clean before the
computation and still holds the AND of the controls at the uncomputation; an
exhaustive classical simulation of the oracle's reversible action confirms
this here. The single pair below shows the construction, with the classically
controlled correction drawn as a conditional block.

In [ ]:
single = QuantumCircuit(3)
single.append(RXGate(math.pi).control(2), [0, 1, 2])
single.z(2)
single.append(RXGate(-math.pi).control(2), [0, 1, 2])
dynamic_single, _, _ = logical_and.measurement_assisted(single, clean_wires=[2])
show_figure(dynamic_single.draw("mpl"), "logical-and-single")
show_expanded(dynamic_single, "logical-and-single")

raw = domain_free.circuit
dynamic, skeleton, info = logical_and.measurement_assisted(raw, clean_wires=workspace)
static_oracle = costs.ft_metrics(costs.clifford_t(raw))
assisted_oracle = logical_and.measurement_assisted_metrics(raw, clean_wires=workspace)
display(
    pd.DataFrame(
        {"static": static_oracle, "measurement-assisted": assisted_oracle}
    ).loc[["t_count", "t_depth", "cx_count", "and_pairs", "measurements", "reaction_depth", "pair_check"]]
)
show_figure(dynamic.draw("mpl", fold=30, idle_wires=False), "oracle-measurement-assisted")
show_expanded(dynamic, "oracle-measurement-assisted")

The dynamic oracle is verified by simulation. A random product state is
prepared on the data and output qubits, and every shot of the dynamic circuit
must reproduce the state produced by the unitary oracle, whatever the
measurement outcomes.

In [ ]:
from qiskit import transpile
from qiskit_aer import AerSimulator

rng = np.random.default_rng(7)
start = QuantumCircuit(raw.num_qubits)
for q in [*inputs, *outputs]:
    start.ry(rng.uniform(0.2, math.pi - 0.2), q)
    start.rz(rng.uniform(0, 2 * math.pi), q)
expected = Statevector(start.compose(raw))
run = dynamic.copy_empty_like()
run.compose(start, inplace=True)
run.compose(dynamic, inplace=True)
run.save_statevector(pershot=True)
simulator = AerSimulator(method="statevector")
result = simulator.run(transpile(run, simulator), shots=8, seed_simulator=7).result()
fidelities = [
    abs(np.vdot(expected.data, np.asarray(state))) ** 2
    for state in result.data()["statevector"]
]
display(pd.DataFrame({"shot": range(len(fidelities)), "fidelity": fidelities}).set_index("shot").T)
assert np.allclose(fidelities, 1)

## 11. Fault-tolerant cost of one complete Grover iteration

The complete search has three components: the preparation $A$, the oracle, and
the diffusion operator $A(2|0\rangle\langle0| - I)A^\dagger$, whose
five-control gate is decomposed with each policy. During diffusion the oracle's
workspace is clean, so clean constructions use it before appending ancillas.
The initial success probability is $6/27$, so one iteration is optimal and
reaches the success probability computed in Section 5.

Every rotation of the preparation is synthesized under a whole-algorithm error
budget of $10^{-3}$, split evenly among all rotations; each costs
$\lceil 3\log_2(1/\varepsilon)\rceil$ T gates (Ross and Selinger, leading term).
Static and measurement-assisted costs are reported separately. The chart that
follows the table separates the synthesized rotations from the exact gates of
each component.

In [ ]:
from logique.circuits.grover import _reflection

width = raw.num_qubits
preparation_full = QuantumCircuit(width)
preparation_full.compose(preparation, inputs, inplace=True)
diffusion_full = QuantumCircuit(width)
diffusion_full.compose(_reflection(preparation), inputs, inplace=True)
p_initial = 6 / 27
iterations = costs.optimal_iterations(p_initial)
success = costs.grover_success(p_initial, iterations)


def priced(circuit, clean, assisted):
    if assisted:
        return logical_and.measurement_assisted_metrics(circuit, clean_wires=clean)
    return costs.ft_metrics(costs.clifford_t(circuit))


rows = []
for method in ("clean_tree", "2_clean_kg24", "noaux_v24"):
    diffusion, info = costs.decompose_mcx(diffusion_full, method, clean_wires=workspace)
    appended = list(range(width, diffusion.num_qubits))
    for assisted in (False, True):
        parts = {
            "preparation": priced(preparation_full, [], False),
            "oracle": priced(raw, workspace, assisted),
            "diffusion": priced(diffusion, workspace + appended, assisted),
        }
        rotations = parts["preparation"]["rotation_count"] + iterations * (
            parts["oracle"]["rotation_count"] + parts["diffusion"]["rotation_count"]
        )
        per_rotation = costs.rotation_t_count(1e-3 / rotations)
        exact = {
            name: part["t_count"] * (1 if name == "preparation" else iterations)
            for name, part in parts.items()
        }
        total = sum(exact.values()) + rotations * per_rotation
        rows.append(
            {
                "MCX policy": method,
                "pricing": "measurement-assisted" if assisted else "static",
                "qubits": diffusion.num_qubits,
                **{f"T {name}, exact gates": value for name, value in exact.items()},
                "rotations": rotations,
                "T per rotation": per_rotation,
                "T rotations": rotations * per_rotation,
                "total T": total,
                "T per success": round(total / success, 1),
            }
        )
iteration_costs = pd.DataFrame(rows).set_index(["MCX policy", "pricing"])
display(iteration_costs)
iteration_costs.to_csv(OUT / "iteration_costs.csv")
print(f"iterations: {iterations}, ideal success: {success:.4f}")

In [ ]:
# Vale et al. is far more expensive and stays in the table above.
shown = iteration_costs.loc[["clean_tree", "2_clean_kg24"]]
parts = {
    "T rotations": "rotations (synthesized)",
    "T preparation, exact gates": "preparation, exact gates",
    "T oracle, exact gates": "oracle",
    "T diffusion, exact gates": "diffusion, exact gates",
}
colors = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]
figure, ax = plt.subplots(figsize=(7.5, 3.6))
labels_x = [f"{m}\n{p}" for m, p in shown.index]
bottom = np.zeros(len(shown))
for (column, label), color in zip(parts.items(), colors):
    values = shown[column].to_numpy(dtype=float)
    ax.bar(labels_x, values, bottom=bottom, color=color, edgecolor="white", linewidth=1.5, label=label, width=0.6)
    bottom += values
for x, total in enumerate(bottom):
    ax.text(x, total + 6, f"{total:.0f}", ha="center", color="#52514e")
ax.set_ylabel("T gates, one complete search")
ax.grid(axis="y", color="#e4e3df")
ax.spines[["top", "right"]].set_visible(False)
ax.legend(frameon=False, ncol=2, loc="upper center", bbox_to_anchor=(0.5, -0.2))
ax.tick_params(axis="x", labelsize=8)
show_figure(figure, "iteration-t-costs")

## 12. Routed metrics on a device model

For NISQ targets, the complete iteration is transpiled to the `FakeTorino`
snapshot (133 qubits, CZ basis) with five fixed seeds; the median and the best
seed are reported. Before routing, each controlled $R_x(\pm\pi)$ is rewritten
exactly as a relative-phase Toffoli and a CZ. The drawing shows where the best
seed placed the logical qubits.

In [ ]:
from qiskit.visualization import plot_circuit_layout

from logique.benchmarks import routing

complete_iteration, _ = costs.decompose_mcx(paper, "clean_tree", clean_wires=workspace)
routed = {
    "oracle": routing.routed_metrics(raw),
    "complete iteration": routing.routed_metrics(complete_iteration),
}
columns = [
    "two_qubit_count",
    "best_two_qubit_count",
    "two_qubit_depth",
    "depth",
    "active_qubits",
    "duration_s",
    "esp",
    "best_seed",
]
display(pd.DataFrame(routed).T[columns])
backend = routing.default_backend()
best = transpile(
    routing.prelower(complete_iteration),
    backend=backend,
    optimization_level=3,
    seed_transpiler=routed["complete iteration"]["best_seed"],
)
show_figure(plot_circuit_layout(best, backend), "torino-layout")

## 13. Record the run

The tables are saved as CSV files and the configuration and versions in
`walkthrough_manifest.json`, beside the figures.

In [ ]:
from logique.benchmarks.provenance import checkout_revision, environment

probabilities.to_csv(OUT / "reflection_probabilities.csv")
labelling.to_csv(OUT / "baseline_labels.csv")
manifest = {
    "instance": "K3, three colors, precolor removed",
    "oracle": "domain-free XAG (regime R3), kick-back form",
    "preparation": "Shukla-Vedula valid-color preparation, matching reflection",
    "rotation_budget": 1e-3,
    "routing": {"backend": routed["oracle"]["backend"], "seeds": routed["oracle"]["seeds"]},
    "versions": environment(),
    "checkout": checkout_revision(ROOT),
}
(OUT / "walkthrough_manifest.json").write_text(json.dumps(manifest, indent=2, default=str) + "\n")
for path in sorted(OUT.rglob("*")):
    if path.is_file():
        print(path.relative_to(OUT))